# Data Quality - Full Pipeline

Runs **all DQ rule sets** against every Bronze table and writes results to
`hospital_catalog.quality.dq_results`.

| Step | What happens |
|------|--------------|
| 1 | Environment setup and path injection |
| 2 | Schema / table bootstrap (idempotent) |
| 3 | Configure which tables to run |
| 4 | Execute DQ loop over all tables |
| 5 | Print per-table summary |
| 6 | Inline DQ analytics (failure rates, critical alerts, quarantine) |

> Set `TABLE_FILTER` in **Step 3** to `"ALL"` to process every table,
> or a specific key (e.g. `"patients"`) to run only that table.

## 1. Setup

In [ ]:
# No external library installs required.
# PySpark and Delta Lake are natively provided by Databricks Runtime.

import sys
from datetime import datetime
import pandas as pd

sys.path.insert(0, '/Workspace/Users/sallam5100@gmail.com/.bundle/hospital_edw/dev/files')
print('Environment ready.')

In [ ]:
from config.quality_config import (
    DQ_CATALOG,
    QUALITY_SCHEMA,
    QUARANTINE_SCHEMA,
    BRONZE_SCHEMA,
    DQ_RESULTS_TABLE,
    DQ_RUNS_TABLE,
)

from src.quality.engine import run_dq

from src.quality.rules import (
    PATIENT_RULES,
    DOCTOR_RULES,
    DEPARTMENT_RULES,
    ENCOUNTER_RULES,
    ADMISSION_RULES,
    DIAGNOSIS_RULES,
    PROCEDURE_RULES,
    LAB_ORDER_RULES,
    LAB_RESULT_RULES,
    MEDICATION_RULES,
    PRESCRIPTION_RULES,
    PHARMACY_TRANSACTION_RULES,
    INVOICE_RULES,
    PAYMENT_RULES,
    INSURANCE_CLAIM_RULES,
)

print('All imports successful.')

## 2. Schema and Table Bootstrap

Creates the `quality` and `quarantine` schemas and the `dq_results` / `dq_runs`
tracking tables if they do not already exist. Safe to re-run.

In [ ]:
bootstrap_statements = [
    f"CREATE SCHEMA IF NOT EXISTS {DQ_CATALOG}.{QUALITY_SCHEMA}",
    f"CREATE SCHEMA IF NOT EXISTS {DQ_CATALOG}.{QUARANTINE_SCHEMA}",
    f"CREATE SCHEMA IF NOT EXISTS {DQ_CATALOG}.{BRONZE_SCHEMA}",
    f"""
    CREATE TABLE IF NOT EXISTS {DQ_RESULTS_TABLE} (
        dq_run_id        STRING    NOT NULL,
        dq_timestamp     TIMESTAMP NOT NULL,
        table_name       STRING    NOT NULL,
        source_branch    STRING,
        source_system    STRING,
        rule_id          STRING    NOT NULL,
        rule_name        STRING    NOT NULL,
        dq_category      STRING    NOT NULL,
        severity         STRING    NOT NULL,
        column_name      STRING,
        total_records    BIGINT    NOT NULL,
        failed_records   BIGINT    NOT NULL,
        passed_records   BIGINT    NOT NULL,
        failure_rate     DOUBLE,
        status           STRING    NOT NULL,
        rule_description STRING
    ) USING DELTA
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DQ_RUNS_TABLE} (
        dq_run_id         STRING    NOT NULL,
        start_timestamp   TIMESTAMP NOT NULL,
        end_timestamp     TIMESTAMP,
        run_status        STRING,
        total_tables      BIGINT,
        total_rules       BIGINT,
        failed_rules      BIGINT,
        critical_failures BIGINT,
        total_records     BIGINT,
        failed_records    BIGINT,
        created_at        TIMESTAMP
    ) USING DELTA
    """,
]

for stmt in bootstrap_statements:
    spark.sql(stmt)

print(f"  OK  {DQ_CATALOG}.{QUALITY_SCHEMA}")
print(f"  OK  {DQ_CATALOG}.{QUARANTINE_SCHEMA}")
print(f"  OK  {DQ_RESULTS_TABLE}")
print(f"  OK  {DQ_RUNS_TABLE}")
print('Bootstrap complete.')

## 3. Table Configuration

`DQ_TABLE_CONFIG` is the single source of truth for which Bronze tables to check
and which rule set(s) to apply. Each entry maps a logical table name to its
Bronze table path, DQ rule list, and source system label.

Set `TABLE_FILTER = "ALL"` to run everything, or a specific key
(e.g. `"patients"`) to run only that table.

In [ ]:
# -- Central DQ Table Registry --
DQ_TABLE_CONFIG = {
    # EMR
    "patients": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.patients",
        "rules":         PATIENT_RULES,
        "source_system": "EMR",
    },
    "doctors": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.doctors",
        "rules":         DOCTOR_RULES,
        "source_system": "EMR",
    },
    "departments": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.departments",
        "rules":         DEPARTMENT_RULES,
        "source_system": "EMR",
    },
    "encounters": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.encounters",
        "rules":         ENCOUNTER_RULES,
        "source_system": "EMR",
    },
    "admissions": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.admissions",
        "rules":         ADMISSION_RULES,
        "source_system": "EMR",
    },
    "diagnoses": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.diagnoses",
        "rules":         DIAGNOSIS_RULES,
        "source_system": "EMR",
    },
    "procedures": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.procedures",
        "rules":         PROCEDURE_RULES,
        "source_system": "EMR",
    },
    # Laboratory
    "lab_orders": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.lab_orders",
        "rules":         LAB_ORDER_RULES,
        "source_system": "Laboratory",
    },
    "lab_results": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.lab_results",
        "rules":         LAB_RESULT_RULES,
        "source_system": "Laboratory",
    },
    # Pharmacy
    "medications": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.medications",
        "rules":         MEDICATION_RULES,
        "source_system": "Pharmacy",
    },
    "prescriptions": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.prescriptions",
        "rules":         PRESCRIPTION_RULES,
        "source_system": "Pharmacy",
    },
    "pharmacy_transactions": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.pharmacy_transactions",
        "rules":         PHARMACY_TRANSACTION_RULES,
        "source_system": "Pharmacy",
    },
    # Billing
    "invoices": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.invoices",
        "rules":         INVOICE_RULES,
        "source_system": "Billing",
    },
    "payments": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.payments",
        "rules":         PAYMENT_RULES,
        "source_system": "Billing",
    },
    "insurance_claims": {
        "bronze_table":  f"{DQ_CATALOG}.{BRONZE_SCHEMA}.insurance_claims",
        "rules":         INSURANCE_CLAIM_RULES,
        "source_system": "Billing",
    },
}

# Set to a specific table key (e.g. "patients") to run only that table.
TABLE_FILTER = "ALL"

tables_to_run = (
    list(DQ_TABLE_CONFIG.keys())
    if TABLE_FILTER == "ALL"
    else [TABLE_FILTER]
)

print(f"Starting DQ run at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Tables to check ({len(tables_to_run)}): {', '.join(tables_to_run)}")

## 4. Execute DQ Pipeline

In [ ]:
results = []

for table_name in tables_to_run:
    cfg           = DQ_TABLE_CONFIG[table_name]
    bronze_table  = cfg["bronze_table"]
    rules         = cfg["rules"]
    source_system = cfg["source_system"]

    print("=" * 70)
    print(f"  Table : {table_name}")
    print(f"  Source: {bronze_table}")
    print(f"  System: {source_system}  |  Rules: {len(rules)}")
    print("=" * 70)

    start_time = datetime.now()

    try:
        df     = spark.table(bronze_table)
        result = run_dq(
            spark         = spark,
            df            = df,
            table_name    = table_name,
            rules         = rules,
            source_system = source_system,
        )

        elapsed  = (datetime.now() - start_time).total_seconds()
        passed   = result.get("passed_rules",      0)
        failed   = result.get("failed_rules",      0)
        critical = result.get("critical_failures", 0)
        total_r  = result.get("total_records",     0)

        results.append({
            "table":             table_name,
            "source_system":     source_system,
            "status":            "SUCCESS",
            "total_rules":       len(rules),
            "passed_rules":      passed,
            "failed_rules":      failed,
            "critical_failures": critical,
            "total_records":     total_r,
            "duration_sec":      round(elapsed, 2),
            "error":             None,
        })

        icon = "OK" if failed == 0 else ("CRITICAL" if critical > 0 else "WARN")
        print(f"  -> [{icon}]  passed={passed}  failed={failed}  "
              f"critical={critical}  rows={total_r:,}  [{elapsed:.2f}s]")

    except Exception as e:
        elapsed = (datetime.now() - start_time).total_seconds()

        results.append({
            "table":             table_name,
            "source_system":     source_system,
            "status":            "FAILED",
            "total_rules":       len(rules),
            "passed_rules":      0,
            "failed_rules":      0,
            "critical_failures": 0,
            "total_records":     0,
            "duration_sec":      round(elapsed, 2),
            "error":             str(e),
        })

        print(f"  -> [ERROR]  {e}")

## 5. Run Summary

In [ ]:
summary_df = pd.DataFrame(results)

success_count  = (summary_df["status"] == "SUCCESS").sum()
error_count    = (summary_df["status"] == "FAILED").sum()
total_rules    = summary_df["total_rules"].sum()
total_passed   = summary_df["passed_rules"].sum()
total_failed   = summary_df["failed_rules"].sum()
total_critical = summary_df["critical_failures"].sum()
total_records  = summary_df["total_records"].sum()

print("\n" + "=" * 70)
print("DATA QUALITY RUN SUMMARY")
print("=" * 70)
print(f"  Tables checked :  {len(summary_df)}")
print(f"  Successful     :  {success_count}")
print(f"  Errored        :  {error_count}")
print(f"  Total rules    :  {total_rules}")
print(f"  Rules passed   :  {total_passed}")
print(f"  Rules failed   :  {total_failed}")
print(f"  Critical fails :  {total_critical}")
print(f"  Total records  :  {total_records:,}")
print("=" * 70)

display(summary_df)

## 6. DQ Analytics

Inline queries against `hospital_catalog.quality.dq_results` for quick inspection.

In [ ]:
# 6a. Rules summary by table / category / severity
display(
    spark.sql(f"""
        SELECT
            table_name,
            dq_category,
            severity,
            COUNT(*)                                              AS rules_checked,
            SUM(CASE WHEN status = 'PASSED' THEN 1 ELSE 0 END)  AS passed_rules,
            SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END)  AS failed_rules
        FROM {DQ_RESULTS_TABLE}
        GROUP BY table_name, dq_category, severity
        ORDER BY table_name, severity
    """)
)

In [ ]:
# 6b. Critical failures (most recent run)
display(
    spark.sql(f"""
        SELECT
            table_name,
            rule_id,
            rule_name,
            source_branch,
            total_records,
            failed_records,
            failure_rate,
            dq_timestamp
        FROM {DQ_RESULTS_TABLE}
        WHERE severity = 'CRITICAL'
          AND status   = 'FAILED'
        ORDER BY dq_timestamp DESC
    """)
)

In [ ]:
# 6c. Worst rules by total failed records
display(
    spark.sql(f"""
        SELECT
            table_name,
            rule_id,
            rule_name,
            dq_category,
            severity,
            SUM(failed_records)  AS total_failed_records,
            AVG(failure_rate)    AS avg_failure_rate
        FROM {DQ_RESULTS_TABLE}
        GROUP BY table_name, rule_id, rule_name, dq_category, severity
        ORDER BY total_failed_records DESC
    """)
)

In [ ]:
# 6d. Quarantine snapshot (patients table as example)
# Adjust the table name for other entities as needed.
try:
    display(
        spark.sql(f"""
            SELECT
                _quarantine_rule_id,
                _quarantine_severity,
                COUNT(*) AS records
            FROM {DQ_CATALOG}.{QUARANTINE_SCHEMA}.patients
            GROUP BY _quarantine_rule_id, _quarantine_severity
            ORDER BY records DESC
        """)
    )
except Exception as e:
    print(f"Quarantine table not yet populated or does not exist: {e}")